# Bronze → Silver: ocorrencias_rede

**Entrada:** `` `v-energy`.bronze.tb_ocorrencias_rede `` (texto bruto, 180.000 linhas).
**Saída:** `` `v-energy`.silver.tb_ocorrencias_rede `` (Delta, tipada e padronizada).
**Grão:** uma linha por ocorrência (`id_ocorrencia` único). Pela definição do case, cada registro é **uma UC interrompida**.

**O que este notebook faz.** Cada passo corresponde a um achado de `docs/auditoria_dados/auditoria_ocorrencias_rede.md`:
1. **Tipa as datas** como `TIMESTAMP_NTZ` (hora local de Recife, sem fuso) e `duracao_min` como inteiro, removendo o sufixo `" min"`.
2. **Padroniza `causa`:** as 4 formas de ausência (`''`, `-`, `N/A`, `NULL`) viram a categoria explícita `NAO INFORMADA`, separada de `NAO IDENTIFICADA`.
3. **Trata as interrupções sobrepostas na mesma UC, dentro de cada mês:** `flag_sobreposicao` e `duracao_efetiva_min`, que conta só os minutos ainda não cobertos por uma interrupção anterior da UC no mesmo mês. Somar `duracao_efetiva_min` de qualquer conjunto de meses, com ou sem o evento atípico, dá o tempo real sem energia, sem contar nada em dobro.
4. **Marca o evento atípico por alimentador e mês** (`flag_evento_atipico`): mês com mais de 5× a base mensal do alimentador. Hoje marca exatamente o evento climático de 2025-07.
5. **Valida antes de gravar:** se uma regra crítica falhar, o notebook para e a Silver atual não é tocada. A validação e a escrita leem a mesma versão da Bronze. Carga completa com `overwrite` (idempotente).

Documentação: `docs/silver/silver_ocorrencias_rede.md`.

In [ ]:
# ruff: noqa: F821
# (spark e display são variáveis globais do notebook no Databricks)
catalog = 'v-energy'
bronze_schema = 'bronze'
silver_schema = 'silver'

bronze_table = f'`{catalog}`.{bronze_schema}.tb_ocorrencias_rede'
silver_table = f'`{catalog}`.{silver_schema}.tb_ocorrencias_rede'

COLUNAS_BRONZE = [
    'id_ocorrencia',
    'alimentador',
    'id_uc',
    'tipo',
    'causa',
    'data_abertura',
    'data_encerramento',
    'duracao_min',
]
TIPOS_VALIDOS = [
    'FALTA DE ENERGIA',
    'DANO EM EQUIPAMENTO',
    'OSCILACAO',
    'SOBRECARGA',
    'FIO PARTIDO',
]
CAUSA_NAO_INFORMADA = 'NAO INFORMADA'
CAUSAS_VALIDAS = [
    'CLIMATICA',
    'SOBRECARGA',
    'NAO IDENTIFICADA',
    'VEGETACAO',
    'TERCEIROS',
    'FALHA DE EQUIPAMENTO',
    CAUSA_NAO_INFORMADA,
]
MARCADORES_AUSENCIA = ['', '-', 'N/A', 'NULL']  # formas de "sem causa" na auditoria (seção 3.2)
FORMATO_DATA_HORA = 'yyyy-MM-dd HH:mm:ss'
FUSO_ORIGEM = 'America/Recife'  # os horários da origem são locais; "agora" é calculado neste fuso
# Evento atípico: mês com mais de LIMIAR_EVENTO_ATIPICO vezes a base mensal do alimentador.
# Com menos de MESES_MINIMOS_SERIE meses, a base do alimentador é a mediana da rede.
LIMIAR_EVENTO_ATIPICO = 5
MESES_MINIMOS_SERIE = 12

In [ ]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{catalog}`.{silver_schema}')

## 1. Leitura da Bronze
Fixa a versão atual da Bronze (`VERSION AS OF`): a validação e a escrita leem exatamente os mesmos dados, mesmo que a Bronze seja recarregada durante a execução. Depois confere se as colunas esperadas existem.

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

versao_bronze = spark.sql(f'DESCRIBE HISTORY {bronze_table} LIMIT 1').first()['version']
df_bronze = spark.sql(f'SELECT * FROM {bronze_table} VERSION AS OF {versao_bronze}')
faltando = set(COLUNAS_BRONZE) - set(df_bronze.columns)
assert not faltando, f'Colunas ausentes na Bronze: {faltando}'

linhas_bronze = df_bronze.count()
print(f'{bronze_table} (versão {versao_bronze}): {linhas_bronze} linhas')

## 2. Padronização e tipagem
- **Datas:** `try_to_timestamp` com o formato único da origem, depois `TIMESTAMP_NTZ`. Os horários são locais e não têm fuso, então o tipo sem fuso garante que nenhuma configuração de sessão desloque as horas. Valor inválido vira nulo e é barrado na validação.
- **`duracao_min`:** extrai os dígitos (aceita `"217"` e `"217 min"`) e converte com `try_cast`.
- **`causa`:** maiúsculas, sem acento, e ausência → `NAO INFORMADA`.
- **`sigla_municipio`** (`REC`, `JAB`…) vem do código do alimentador, e **`ano_mes`** do mês de abertura.
- As duplicatas exatas (as mesmas colunas da Bronze) são removidas depois da padronização. A Bronze atual não tem nenhuma, mas a regra protege as próximas cargas.

In [ ]:
def texto(coluna):
    """Remove espaços nas pontas, converte para maiúsculas e tira acentos."""
    return F.translate(
        F.upper(F.trim(F.col(coluna))), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ', 'AAAAAEEEEIIIIOOOOOUUUUC'
    )


def para_data_hora(coluna):
    return F.try_to_timestamp(F.trim(F.col(coluna)), F.lit(FORMATO_DATA_HORA)).cast('timestamp_ntz')


causa = texto('causa')

df_padronizado = (
    df_bronze.select(
        texto('id_ocorrencia').alias('id_ocorrencia'),
        texto('id_uc').alias('id_uc'),
        texto('alimentador').alias('alimentador'),
        texto('tipo').alias('tipo'),
        F.when(causa.isNull() | causa.isin(MARCADORES_AUSENCIA), F.lit(CAUSA_NAO_INFORMADA))
        .otherwise(causa)
        .alias('causa'),
        para_data_hora('data_abertura').alias('data_abertura'),
        para_data_hora('data_encerramento').alias('data_encerramento'),
        # "217", "217 min" ou "217 MIN" → "217"; outro formato vira nulo e é barrado na validação
        F.regexp_extract(F.col('duracao_min'), r'(?i)^\s*(\d+)\s*(min)?\s*$', 1).alias(
            '_duracao_texto'
        ),
        'source_file',
        'ingestion_datetime',
    )
    .withColumn('duracao_min', F.expr("try_cast(nullif(_duracao_texto, '') AS INT)"))
    .withColumn('sigla_municipio', F.regexp_extract('alimentador', r'^ALM-([A-Z]{3})-\d{3}$', 1))
    .withColumn('ano_mes', F.date_format('data_abertura', 'yyyy-MM'))
    .drop('_duracao_texto')
)

# Duplicata exata: as mesmas colunas da Bronze, depois da padronização
df_unico = df_padronizado.dropDuplicates(COLUNAS_BRONZE)

## 3. Interrupções sobrepostas na mesma UC
Para cada ocorrência, `maior_fim_anterior` é o **maior encerramento** entre as ocorrências anteriores da mesma UC **abertas no mesmo mês**. Comparar só com a imediatamente anterior não basta, porque uma interrupção longa pode conter uma curta e uma terceira.
- `flag_sobreposicao`: a ocorrência começa antes de `maior_fim_anterior`, ou seja, a UC já estava sem energia.
- `duracao_efetiva_min`: minutos desta ocorrência que **ainda não** estavam cobertos (`encerramento − maior_fim_anterior`, ou zero). Para ocorrências sem sobreposição, é igual a `duracao_min`.

Assim, `SUM(duracao_efetiva_min)` é o tempo real sem energia (união dos intervalos, base do DEC), e `COUNT_IF(NOT flag_sobreposicao)` é o número de interrupções distintas (base do FEC).

**Por que dentro do mês:** a Gold soma por mês e escolhe se inclui o evento atípico. Se uma ocorrência de outro mês, ou de um mês de evento atípico, cobrisse esta, os minutos ficariam com a outra e sumiriam quando a Gold tirasse aquele mês da conta. A janela é por UC, alimentador e mês, e o flag de evento atípico vale para o alimentador no mês inteiro, então ela nunca mistura ocorrências com e sem o flag. O custo é pequeno: uma interrupção que atravessa a virada do mês e cobre outra do mês seguinte conta nos dois meses (na extração atual, 1 caso e 34 min).

In [ ]:
# Janela por UC e mês: o flag de evento atípico vale para o alimentador no mês inteiro, então a
# Gold pode somar qualquer conjunto de meses, com ou sem o evento, sem perder minutos
anteriores_no_mes = (
    Window.partitionBy('id_uc', 'alimentador', 'ano_mes')
    .orderBy('data_abertura', 'data_encerramento', 'id_ocorrencia')
    .rowsBetween(Window.unboundedPreceding, -1)
)
minutos_nao_cobertos = F.expr('timestampdiff(MINUTE, _maior_fim_anterior, data_encerramento)')

df_sobreposicao = (
    df_unico.withColumn('_maior_fim_anterior', F.max('data_encerramento').over(anteriores_no_mes))
    .withColumn(
        'flag_sobreposicao',
        F.coalesce(F.col('data_abertura') < F.col('_maior_fim_anterior'), F.lit(False)),
    )
    .withColumn(
        'duracao_efetiva_min',
        F.when(F.col('flag_sobreposicao'), F.greatest(F.lit(0), minutos_nao_cobertos))
        .otherwise(F.col('duracao_min'))
        .cast('int'),
    )
    .drop('_maior_fim_anterior')
)

## 4. Evento atípico por alimentador e mês
Para cada alimentador, conta as ocorrências de cada mês do calendário **entre o primeiro e o último mês em que ele aparece**. A grade é gerada com `sequence`, então um mês sem nenhuma ocorrência entra com zero. Um mês é de **evento atípico** quando passa de `LIMIAR_EVENTO_ATIPICO` vezes a **base mensal** do alimentador:
- **base** = mediana mensal do próprio alimentador, mas nunca abaixo da **mediana da rede** (todas as combinações alimentador × mês). O piso impede que um alimentador quase sem ocorrências tenha marcado qualquer mês com 6 ou mais.
- Com menos de `MESES_MINIMOS_SERIE` meses, a base é a mediana da rede. Numa série curta, a mediana do próprio alimentador inclui o evento: com os meses [30, 6.887], ela daria 3.458 e o evento passaria.

O intervalo de cada alimentador importa para os que entram ou saem no meio do período: se os meses em que ele ainda não existia (ou já tinha saído) entrassem com zero, a mediana dele iria a zero.

Na extração atual, os 122 alimentadores aparecem nos 48 meses, a mediana da rede é 28 e as medianas dos alimentadores vão de 25 a 29,5. Os 3 alimentadores do evento climático de 2025-07 ficam entre **93× e 246×** a base, e todos os outros meses ficam em no máximo **2,3×**. O limite de 5× separa os dois grupos com folga, sem data fixa no código. O flag não remove nada: a Gold decide se calcula DEC/FEC com ou sem esses meses.

In [ ]:
por_alimentador_mes = df_sobreposicao.groupBy('alimentador', 'ano_mes').agg(
    F.count('*').alias('ocorrencias')
)

# Calendário completo entre o primeiro e o último mês de cada alimentador
primeiro, ultimo = F.to_date(F.min('ano_mes'), 'yyyy-MM'), F.to_date(F.max('ano_mes'), 'yyyy-MM')
grade = (
    por_alimentador_mes.groupBy('alimentador')
    .agg(F.sequence(primeiro, ultimo, F.expr('INTERVAL 1 MONTH')).alias('_meses'))
    .select('alimentador', F.explode('_meses').alias('_mes'))
    .select('alimentador', F.date_format('_mes', 'yyyy-MM').alias('ano_mes'))
)
serie_mensal = grade.join(por_alimentador_mes, ['alimentador', 'ano_mes'], 'left').fillna(
    0, ['ocorrencias']
)

# Base do alimentador: a própria mediana, nunca abaixo da mediana da rede (nem de 1)
mediana_rede = serie_mensal.agg(F.median('ocorrencias')).first()[0]
piso = F.lit(max(mediana_rede or 0, 1))
base = (
    serie_mensal.groupBy('alimentador')
    .agg(F.median('ocorrencias').alias('mediana_mensal'), F.count('*').alias('meses'))
    .withColumn(
        'base_mensal',
        F.when(F.col('meses') < MESES_MINIMOS_SERIE, piso).otherwise(
            F.greatest('mediana_mensal', piso)
        ),
    )
)

meses_avaliados = serie_mensal.join(base, 'alimentador').withColumn(
    'flag_evento_atipico', F.col('ocorrencias') > LIMIAR_EVENTO_ATIPICO * F.col('base_mensal')
)
meses_atipicos = meses_avaliados.where('flag_evento_atipico').select(
    'alimentador', 'ano_mes', 'flag_evento_atipico'
)

df_tratado = (
    df_sobreposicao.join(F.broadcast(meses_atipicos), ['alimentador', 'ano_mes'], 'left')
    .fillna(False, ['flag_evento_atipico'])
    .withColumn('silver_datetime', F.current_timestamp())
)

## 5. Validação antes de gravar
Cada regra conta as linhas que a **violam**, e todas são calculadas numa única passada.
- **crítica:** interrompe o notebook antes do `overwrite`, e a Silver atual não é tocada. Antes de parar, o notebook mostra até 20 linhas que violam cada regra.
- **aviso:** é exibida, mas não bloqueia a carga.

"Agora" é a hora atual em `FUSO_ORIGEM`, o mesmo fuso das datas da origem. A sessão do Databricks costuma estar em UTC, 3 horas à frente.

In [ ]:
# Agora no fuso da origem, como TIMESTAMP_NTZ (a sessão costuma estar em UTC, 3 horas à frente)
agora = F.expr(f"convert_timezone('{FUSO_ORIGEM}', current_timestamp())")

df_check = df_tratado.withColumn(
    '_repeticoes_id', F.count('*').over(Window.partitionBy('id_ocorrencia'))
)

duracao_pelas_datas = F.expr('timestampdiff(MINUTE, data_abertura, data_encerramento)')

REGRAS = [  # (regra, condição que identifica uma linha que VIOLA a regra, crítica?)
    ('id_ocorrencia repetido após remover duplicatas exatas', F.col('_repeticoes_id') > 1, True),
    (
        'id_ocorrencia ou id_uc fora do padrão OC…/UC… + dígitos',
        F.col('id_ocorrencia').isNull()
        | ~F.col('id_ocorrencia').rlike(r'^OC\d+$')
        | F.col('id_uc').isNull()
        | ~F.col('id_uc').rlike(r'^UC\d+$'),
        True,
    ),
    (
        'alimentador fora do padrão ALM-AAA-999',
        F.col('sigla_municipio').isNull() | (F.col('sigla_municipio') == ''),
        True,
    ),
    ('tipo fora do domínio', F.col('tipo').isNull() | ~F.col('tipo').isin(TIPOS_VALIDOS), True),
    ('causa fora do domínio', ~F.col('causa').isin(CAUSAS_VALIDAS), True),
    ('data_abertura vazia ou não convertida', F.col('data_abertura').isNull(), True),
    ('data_encerramento vazia ou não convertida', F.col('data_encerramento').isNull(), True),
    (
        'encerramento anterior à abertura',
        F.col('data_encerramento') < F.col('data_abertura'),
        True,
    ),
    (
        'abertura ou encerramento no futuro',
        F.greatest('data_abertura', 'data_encerramento') > agora,
        True,
    ),
    ('duracao_min vazia ou não convertida', F.col('duracao_min').isNull(), True),
    (
        'duracao_min diferente de encerramento − abertura',
        F.col('duracao_min') != duracao_pelas_datas,
        True,
    ),
    (
        'duracao_efetiva_min fora do intervalo [0, duracao_min]',
        F.col('duracao_efetiva_min').isNull()
        | (F.col('duracao_efetiva_min') < 0)
        | (F.col('duracao_efetiva_min') > F.col('duracao_min')),
        True,
    ),
]

contagens = df_check.agg(
    F.count('*').alias('linhas'),
    F.sum('duracao_min').alias('minutos_somados'),
    F.sum('duracao_efetiva_min').alias('minutos_efetivos'),
    F.count_if('flag_sobreposicao').alias('sobrepostas'),
    F.count_if('flag_evento_atipico').alias('em_evento_atipico'),
    *[F.count_if(condicao).alias(f'regra_{i}') for i, (_, condicao, _) in enumerate(REGRAS)],
).first()

linhas_silver = contagens['linhas']
somados, efetivos = contagens['minutos_somados'], contagens['minutos_efetivos']
resultado = [
    (regra, int(contagens[f'regra_{i}']), 'crítica' if critica else 'aviso')
    for i, (regra, _, critica) in enumerate(REGRAS)
]
display(
    spark.createDataFrame(
        [
            (r, v, n, 'OK' if v == 0 else ('FALHA' if n == 'crítica' else 'ATENÇÃO'))
            for r, v, n in resultado
        ],
        'regra string, violacoes long, nivel string, status string',
    )
)
print(
    f'Bronze: {linhas_bronze} linhas | Silver: {linhas_silver} linhas | '
    f'duplicatas removidas: {linhas_bronze - linhas_silver}'
)
print(
    f'Sobrepostas: {contagens["sobrepostas"]} | minutos somados: {somados} | '
    f'minutos efetivos: {efetivos} | em dobro: {somados - efetivos}'
)
print(f'Ocorrências em evento atípico: {contagens["em_evento_atipico"]}')
print(f'Mediana da rede (alimentador × mês): {mediana_rede}')
display(
    meses_avaliados.where('flag_evento_atipico')
    .select('alimentador', 'ano_mes', 'ocorrencias', 'mediana_mensal', 'base_mensal')
    .orderBy('ano_mes', 'alimentador')
)

falhas = [f'{r}: {v} linha(s)' for r, v, n in resultado if v and n == 'crítica']
for (regra, condicao, critica), (_, violacoes, _) in zip(REGRAS, resultado):
    if critica and violacoes:
        print(f'Exemplos de linhas que violam: {regra}')
        display(df_check.where(condicao).limit(20))
assert not falhas, 'Silver de ocorrências reprovada na validação:\n- ' + '\n- '.join(falhas)

## 6. Escrita na Silver
Os comentários de coluna vão no schema (`withMetadata`) e são gravados no mesmo commit da carga. O comentário da tabela só é reescrito quando muda. O texto vai escapado (`literal_sql`), então um apóstrofo no comentário não quebra a célula depois do `overwrite`.

In [ ]:
COMENTARIOS = {
    'id_ocorrencia': (
        'Identificador da ocorrência (chave da tabela). Cada ocorrência é uma UC interrompida.'
    ),
    'id_uc': 'Unidade consumidora interrompida.',
    'alimentador': 'Alimentador de média tensão (ALM-<sigla do município>-<nnn>).',
    'sigla_municipio': 'Sigla do município extraída do alimentador (REC, JAB, OLI...).',
    'tipo': (
        'Tipo da ocorrência (FALTA DE ENERGIA, FIO PARTIDO, OSCILACAO, SOBRECARGA, '
        'DANO EM EQUIPAMENTO).'
    ),
    'causa': (
        'Causa registrada; NAO INFORMADA quando a origem não trouxe causa '
        '(diferente de NAO IDENTIFICADA).'
    ),
    'data_abertura': 'Início da interrupção, hora local de Recife (TIMESTAMP_NTZ, sem fuso).',
    'data_encerramento': 'Fim da interrupção, hora local de Recife (TIMESTAMP_NTZ, sem fuso).',
    'ano_mes': 'Mês de abertura no formato yyyy-MM.',
    'duracao_min': 'Duração registrada em minutos (= encerramento - abertura).',
    'duracao_efetiva_min': (
        'Minutos desta ocorrência não cobertos por ocorrência anterior da mesma UC no mesmo mês. '
        'Use SUM desta coluna para o tempo sem energia (DEC).'
    ),
    'flag_sobreposicao': (
        'True se a ocorrência começou enquanto a UC já estava sem energia por outra ocorrência '
        'aberta no mesmo mês. Para contar interrupções (FEC), use as linhas com False.'
    ),
    'flag_evento_atipico': (
        'True se a ocorrência foi aberta num mês de evento atípico do alimentador: mais de 5x '
        'a base mensal dele (hoje: evento climático de 2025-07).'
    ),
    'source_file': 'Arquivo de origem na landing.',
    'ingestion_datetime': 'Momento da carga na Bronze.',
    'silver_datetime': 'Momento da carga na Silver.',
}
COLUNAS_SILVER = list(COMENTARIOS)
COMENTARIO_TABELA = (
    'Silver: ocorrências de rede tipadas e padronizadas, uma linha por UC interrompida, com '
    'tratamento de sobreposição e marcação de evento atípico. '
    'Regras em docs/silver/silver_ocorrencias_rede.md.'
)


def literal_sql(texto):
    """Texto entre aspas simples para SQL, com barra invertida e apóstrofo escapados."""
    return "'" + texto.replace('\\', '\\\\').replace("'", "\\'") + "'"


df_final = df_tratado.select(*COLUNAS_SILVER)
for coluna, comentario in COMENTARIOS.items():
    df_final = df_final.withMetadata(coluna, {'comment': comentario})

(
    df_final.write.format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(silver_table)
)

if spark.sql(f'DESCRIBE DETAIL {silver_table}').first()['description'] != COMENTARIO_TABELA:
    spark.sql(f'COMMENT ON TABLE {silver_table} IS {literal_sql(COMENTARIO_TABELA)}')

In [ ]:
# Depois de gravar: a tabela tem as colunas esperadas e o volume validado acima
tabela = spark.table(silver_table)
linhas_tabela = tabela.count()

assert tabela.columns == COLUNAS_SILVER, f'Colunas inesperadas: {tabela.columns}'
assert linhas_tabela == linhas_silver, (
    f'Tabela com {linhas_tabela} linhas, esperado {linhas_silver}'
)
print(f'OK: {linhas_tabela} ocorrências carregadas em {silver_table}')

## Evidências: antes × depois e uso na Gold
As consultas abaixo comparam a Bronze lida no passo 1 (mesma versão) com a Silver e mostram como a Gold deve usar as colunas novas. O `id_ocorrencia` da Bronze passa pela mesma padronização da Silver antes do join. Elas não alteram dados.

In [ ]:
# Bronze lida no passo 1 (mesma versão), com o id padronizado como na Silver
df_bronze.withColumn('id_silver', texto('id_ocorrencia')).createOrReplaceTempView('bronze_lida')
b, s = 'bronze_lida', silver_table

# 1. causa: valores da Bronze → Silver
display(
    spark.sql(f"""
  SELECT coalesce(b.causa, '<null>') AS causa_bronze, s.causa AS causa_silver,
         COUNT(*) AS ocorrencias
  FROM {b} b JOIN {s} s ON b.id_silver = s.id_ocorrencia
  GROUP BY 1, 2 ORDER BY 3 DESC""")
)

# 2. duracao_min: com sufixo "min" (em qualquer caixa) ou só dígitos na Bronze → inteiro na Silver
display(
    spark.sql(f"""
  SELECT CASE WHEN b.duracao_min RLIKE '(?i)min' THEN 'com sufixo min' ELSE 'só dígitos' END
           AS formato_bronze,
         COUNT(*) AS ocorrencias, SUM(s.duracao_min) AS minutos_silver
  FROM {b} b JOIN {s} s ON b.id_silver = s.id_ocorrencia
  GROUP BY 1""")
)

# 3. Tempo sem energia: soma direta × efetiva (sem contar sobreposição em dobro)
display(
    spark.sql(f"""
  SELECT COUNT(*) AS ocorrencias,
         COUNT_IF(NOT flag_sobreposicao) AS interrupcoes_distintas,
         SUM(duracao_min) AS minutos_somados,
         SUM(duracao_efetiva_min) AS minutos_efetivos,
         SUM(duracao_min) - SUM(duracao_efetiva_min) AS minutos_em_dobro
  FROM {s}""")
)

# 4. Prévia da Gold: numeradores de DEC/FEC por alimentador, com e sem o evento atípico
#    (o denominador, nº de UCs do alimentador, vem do cadastro)
display(
    spark.sql(f"""
  SELECT alimentador,
         ROUND(SUM(duracao_efetiva_min) / 60, 1) AS horas_sem_energia,
         COUNT_IF(NOT flag_sobreposicao) AS interrupcoes,
         ROUND(SUM(CASE WHEN NOT flag_evento_atipico THEN duracao_efetiva_min END) / 60, 1)
           AS horas_sem_evento_atipico,
         COUNT_IF(NOT flag_sobreposicao AND NOT flag_evento_atipico)
           AS interrupcoes_sem_evento_atipico
  FROM {s} GROUP BY 1 ORDER BY 2 DESC LIMIT 10""")
)